# PerfumeInsightLab
## Notebook 02 : Thematic EDA
--------------------------------------------------------------------  
#### Part of the multi-notebook EDA workflow : 01 (Data Overview) → 02 → 03 (Quantitative EDA) → 04 (Storytelling & Insights)
#### 1. Import librairies
#### 2. Load dataset  
#### 3. Feature engineering
#### 4. Data quality verification
#### 5. Exploration & Analysis
- CATEGORICAL COLUMNS EXPLORATION  
- PERFUMERS EXPLORATION  
- PERFUMES EXPLORATION   
- OLFACTORY EXPLORATION  

In [ ]:
# ------------------------------------------------------------------
# 1. Import libraries
# ------------------------------------------------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import unidecode

# ------------------------------------------------------------------
# 2. Load dataset
# ------------------------------------------------------------------
df = pd.read_csv("../data/fra_cleaned_v2.csv", encoding="ISO-8859-1", sep=";")

In [ ]:
# ------------------------------------------------------------------
# 3. Feature engineering
# ------------------------------------------------------------------
# Weighted Rating calculation (corrected metric)

df_rated = df.dropna(subset=['Rating Value', 'Rating Count']).copy()      # Filter data to calculate the mean (C) and the threshold (m) accurately

                                                                          # Define the parameters for the weighted rating formula :
C = df_rated['Rating Value'].mean()              # C = Mean rating across the entire rated sample (N=509)
m = df_rated['Rating Count'].quantile(0.90)      # m = Minimum number of votes required (using the 90th percentile of Rating Count)
m = max(100, m)                                  # Ensure 'm' is at least 100 for credibility

print(f"GLOBAL MEAN RATING (C) : {C:.2f}")       # print(f"Weighted Rating calculated. Global C: {C:.2f}, Threshold m: {m:.0f}")
print(f"MIN VOTE THRESHOLD (m) : {m:.0f}")
                                                 
def weighted_rating(row, m=m, C=C):                                       # Define the Weighted Rating function
    v = row['Rating Count']
    R = row['Rating Value']
    
    if pd.isna(v) or pd.isna(R):       # Handle NaN values for non-rated perfumes
        return np.nan
     
    return (v*R + m*C) / (v + m)       # Formula: (v*R + m*C) / (v+m)

df['weighted_rating'] = df.apply(weighted_rating, axis=1)                  # Apply the function to create the new column

print("\nTHE 'weighted_rating' COLUMN HAS BEEN CALCULATED ON THE ENTIRE DATAFRAME")
print("THIS COLUMN WILL NOW BE USED FOR ALL SUBSEQUENT PERFORMANCE ANALYSES")

top_weighted = df.sort_values('weighted_rating', ascending=False).head(10).round(3)
print("\nTOP 10 PERFUMES BASED ON WEIGHTED RATING (corrected for popularity bias) :\n")
display(top_weighted[['Perfume', 'Brand', 'Rating Value', 'Rating Count', 'weighted_rating']])

In [ ]:
# ------------------------------------------------------------------
# 4. Data quality verification
# ------------------------------------------------------------------
print("COLUMN TYPES :")              # Checking data types for each column
df.dtypes

In [ ]:
df.columns

In [ ]:
print("MISSING VALUES PER COLUMN (only showing >0) :")
missing = df.isnull().sum()                
missing[missing > 0]

In [ ]:
# ------------------------------------------------------------------
# 5. Exploration & Analysis
# ------------------------------------------------------------------

In [ ]:
# ---------------------------------------------------------- CATEGORICAL COLUMNS EXPLORATION ----------------------------------------------------------
# -----------------------------------------------------------------------------------------------------------------------------------------------------

categorical_cols = ['Perfume', 'Brand', 'Perfumer', 'Country', 'Gender']         # Unique values per column 
                    
for col in categorical_cols:
    print(f"\nTOP 5 UNIQUE VALUES IN '{col}' :")
    display(df[col].value_counts().head(5))

In [ ]:
categorical_cols = ['Top Notes', 'Middle Notes', 'Base Notes']

for col in categorical_cols:
    print(f"\nTOP 5 UNIQUE VALUES IN '{col}' :")
    display(df[col].value_counts().head(5))

In [ ]:
top50 = df.sort_values('weighted_rating', ascending=False).head(50)

for note_type in ['Top Notes', 'Middle Notes', 'Base Notes']:
    print(f"\nMOST COMMON {note_type} :")
    notes_series = top50[note_type].dropna().str.split(',').explode().str.strip()
    display(notes_series.value_counts().head(5))

In [ ]:
categorical_cols = ['mainaccord1', 'mainaccord2', 'mainaccord3', 'mainaccord4', 'mainaccord5']
                    
for col in categorical_cols:
    print(f"\nTOP 5 UNIQUE VALUES IN '{col}' :")
    display(df[col].value_counts().head(5))

In [ ]:
for gender in df['Gender'].dropna().unique():                                     
    print(f"\nTOP 5 MAINACCORDS FOR {gender} PERFUMES :")
    gender_data = df[df['Gender'] == gender]
    mainaccords = gender_data[['mainaccord1','mainaccord2','mainaccord3','mainaccord4','mainaccord5']]
    top_accords = pd.Series(mainaccords.values.ravel()).value_counts().head(5)
    display(top_accords)

In [ ]:
unique_countries = df['Country'].nunique()                       # To understand dataset diversity and geographical coverage 
print(f"NUMBER OF UNIQUE COUNTRIES : {unique_countries}")                           

In [ ]:
country_mainaccords = (
    df.melt(id_vars='Country', value_vars=['mainaccord1','mainaccord2','mainaccord3','mainaccord4','mainaccord5'])
    .dropna().groupby(['Country','value']).size().reset_index(name='count')
)

top_by_country = (
    country_mainaccords.sort_values(['Country','count'], ascending=[True, False])
    .groupby('Country').head(3)
)

display(top_by_country)

In [ ]:
df.groupby(['Country','Brand']).size().sort_values(ascending=False).head(10)           # Which brands dominate by country ?

In [ ]:
numerical_cols = ['Rating Count', 'Year']                             # Numerical columns statistics
print("DESCRIPTIVE STATISTICS (numerical columns) :\n")
display(df[numerical_cols].describe().round(0).astype(int))

In [ ]:
# --------------------------------------------------------------- PERFUMERS EXPLORATION ---------------------------------------------------------------
# -----------------------------------------------------------------------------------------------------------------------------------------------------

df['Perfumer'] = df['Perfumer'].str.lower().str.strip()            # Lowercase and strip whitespaces for consistency
df['Perfumer'].head(10)

In [ ]:
df['Perfumer'].tail(10)

In [ ]:
df['Perfumer'].value_counts().head(10)

In [ ]:
df['Perfumer'].value_counts().tail(10)

In [ ]:
unique_perfumer_count = df.loc[df['Perfumer'] != 'unknown', 'Perfumer'].nunique() 
print(f"NBER OF UNIQUE PERFUMERS (excluding 'unknown') : {unique_perfumer_count}")

unknown_count = df['Perfumer'].value_counts().get('unknown', 0) 
total_count = len(df)
unknown_percent = (unknown_count / total_count) * 100
print(f"PERCENTAGE OF 'unknown' PERFUMERS : {unknown_percent:.2f}%")

In [ ]:
unknown_ratio = df.groupby('Brand')['Perfumer'].apply(                         # % of perfumes by an “unknown” perfumer per brand
    lambda x: (x == 'unknown').mean() * 100).sort_values(ascending=False)
unknown_ratio.head(5)

In [ ]:
top_perfumers = df['Perfumer'].value_counts().head(10)                      
print("TOP 10 MOST PRODUCTIVE PERFUMERS (by nber of perfumes) :\n")
display(top_perfumers)

In [ ]:
# Perfumers who collaborated with multiple brands (potential industry influencers)               
df[['Perfumer','Brand']].dropna().drop_duplicates().groupby('Perfumer').nunique()['Brand'].sort_values(ascending=False).head(10)

In [ ]:
top_perf_multi = (
    df[['Perfumer','Brand']]
    .dropna().drop_duplicates().groupby('Perfumer')['Brand'].nunique().sort_values(ascending=False).head(10)
)

top_perf_multi.plot(kind='barh', figsize=(10,5), color='slateblue')
plt.title("PERFUMERS WORKING ACROSS MULTIPLE BRANDS\n")
plt.xlabel("Nber of Brands")
plt.show()

In [ ]:
# Defining a reliable dataset (filtered to only include perfumes with a Rating Count of 1000 or more)
df_filtered = df[df['Rating Count'] >= 1000].copy()
print(f"TOTAL PERFUMES IN THE RELIABLE SUBSET (Count >= 1000) : {df_filtered.shape[0]}")

In [ ]:
df_filtered = df[df['Rating Count'] >= 1000].copy()                           # The line create the filtered dataset (from your previous cell):
                                                                   
df_filtered['weighted_rating'] = df_filtered.apply(weighted_rating, axis=1)   # CRITICAL STEP : Add the 'weighted_rating' column to df_filtered.

top_50_success = (                                                            # Sorted calculation, now using the column on df_filtered:
    df_filtered.sort_values(by='weighted_rating', ascending=False)
    .head(50)
)

perfumers_list = top_50_success['Perfumer'].str.split(',').explode()
perfumer_counts = perfumers_list.value_counts()

print("TOP 10 'HIT-MAKERS' (Perfumers on the Top 50 Filtered List, by WEIGHTED RATING) :\n")
display(perfumer_counts.head(10))

In [ ]:
plt.figure(figsize=(10, 5))
perfumer_counts.head(10).sort_values().plot(kind='barh', color='skyblue')
plt.title('TOP 10 PERFUMERS IN THE TOP 50 MOST SUCCESSFUL PERFUMES (filtered)')
plt.xlabel('Nber of Perfumes in Top 50')
plt.ylabel('Perfumer')
plt.show()

In [ ]:
perfumer_name = 'kurkdjian'
perfumer_perfumes = df[df['Perfumer'].str.lower().str.contains(perfumer_name)]
print(f"NUMBER OF PERFUMES BY {perfumer_name}: {len(perfumer_perfumes)}")
display(perfumer_perfumes[['Perfume','Brand','Year']])

In [ ]:
perfumer_name = 'serge lutens'
lutens_perfumes = df[df['Perfumer'].str.lower().str.contains(perfumer_name)]
print(f"NUMBER OF PERFUMES BY {perfumer_name} : {len(lutens_perfumes)}\n")
display(lutens_perfumes[['Perfume','Brand','Year']])

In [ ]:
df['Perfumer'] = df['Perfumer'].str.lower().str.strip()                  # Normalize existing 'Perfumer' column

lutens_perfumes = df[df['Perfumer'].str.contains('lutens', na=False)]    # Check all entries containing 'lutens'
display(lutens_perfumes[['Perfume','Brand','Year']])

In [ ]:
def find_perfumer(df, perfumer_name): 
    perfumer_name = perfumer_name.lower().strip()                                  # Normalize the perfumer_name
    
                                                                                   # Split merged perfumers into list and normalize
    df['Perfumer_list'] = df['Perfumer'].str.split(',').apply(lambda x: [p.strip().lower() for p in x]) 
    
    result = df[df['Perfumer_list'].apply(lambda lst: perfumer_name in lst)]       # Filter rows where perfumer_name is in the list
    return result

lutens_perfumes = find_perfumer(df, "serge lutens")
print(f"NUMBER OF PERFUMES BY SERGE LUTENS : {len(lutens_perfumes)}\n")
display(lutens_perfumes[['Perfume', 'Brand', 'Year', 'Rating Value', 'weighted_rating']])

In [ ]:
ropion_perfumes = find_perfumer(df, "dominique ropion")
print(f"NUMBER OF PERFUMES BY DOMINIQUE ROPION : {len(ropion_perfumes)}\n")
display(ropion_perfumes[['Perfume', 'Brand', 'Year', 'Rating Value', 'weighted_rating']])

In [ ]:
cresp_perfumes = find_perfumer(df, "olivier cresp")
print(f"NUMBER OF PERFUMES BY OLIVIER CRESP : {len(cresp_perfumes)}\n")
display(cresp_perfumes[['Perfume', 'Brand', 'Year', 'Rating Value', 'weighted_rating']])

In [ ]:
# --------------------------------------------------------------- PERFUMES EXPLORATION ----------------------------------------------------------------
# -----------------------------------------------------------------------------------------------------------------------------------------------------

df['Rating Value'] = pd.to_numeric(df['Rating Value'], errors='coerce')

top_rated_perfumes = df[['Perfume', 'Brand', 'Year', 'Rating Value', 'Rating Count', 'weighted_rating']].dropna()
top_rated_perfumes = top_rated_perfumes.sort_values(by='weighted_rating', ascending=False).head(10).round(3)

print("TOP 10 PERFUMES (by WEIGHTED RATING) :")
display(top_rated_perfumes)

In [ ]:
oldest_perfumes = df[['Perfume', 'Brand', 'Year']].query("Year > 0").sort_values(by='Year', ascending=True).head(10)
print("OLDEST PERFUMES STILL ON RECORD (ignoring Year = 0) :\n")
display(oldest_perfumes)

In [ ]:
releases_since_year_x = (
    df[df['Year'] >= 2010]                 
    .groupby('Year')['Perfume']                # group by year
    .nunique()                                 # count unique perfume names
    .reset_index(name='Number_of_Perfumes')    # clean output
    .sort_values('Year', ascending=True)
)

print("NUMBER OF PERFUMES RELEASED PAER YEAR SINCE 2010 :\n")
display(releases_since_year_x)

In [ ]:
df['Year'] = df['Year'].astype(float)
releases_since_2010 = (
    df[df['Year'] >= 2010]
    .groupby('Year')['Perfume']   
    .nunique()         
    .reset_index(name='Number_of_Perfumes')
)

plt.figure(figsize=(8,4))
plt.plot(releases_since_2010['Year'], releases_since_2010['Number_of_Perfumes'], marker='o')
plt.title("NUMBER OF UNIQUE PERFUME RELEASES PER YEAR (since 2010)\n")
plt.xlabel("Year")
plt.ylabel("Nber of Unique Perfumes")
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
df.groupby(['Country','Brand']).size().reset_index(name='Perfume Count').sort_values('Perfume Count', ascending=False).head(10)

In [ ]:
def search_perfume(keyword):                                            # To quickly find any mention of a word
    return df[df[['Perfume','Perfumer', 'Brand']].apply(lambda x: x.str.contains(keyword, case=False, na=False)).any(axis=1)]

display(search_perfume("lutens"))

In [ ]:
def highlight_match(val, keyword):
    if isinstance(val, str) and keyword.lower() in val.lower():
        return 'background-color: #ffe599'          # pale yellow highlight
    return ''

def search_perfume(keyword):                        # Search perfumes or perfumers containing a keyword (case-insensitive)
    result = df[df[['Perfume','Perfumer','Brand']]
                .apply(lambda x: x.str.contains(keyword, case=False, na=False))
                .any(axis=1)]
    print(f"{len(result)} RESULT(S) FOUND FOR '{keyword}'\n")

    styled = result[['Perfume','Brand','Perfumer','Rating Value','weighted_rating','Gender']].style.map(lambda v: highlight_match(v, keyword))
    display(styled)
    return result

In [ ]:
search_perfume("lutens")

In [ ]:
# ----------------------------------------------------------------- OLFACTORY EXPLORATION -------------------------------------------------------------
# -----------------------------------------------------------------------------------------------------------------------------------------------------

all_notes = pd.concat([                      # Concatenate all notes columns and normalize text
    df['Top Notes'].dropna(),
    df['Middle Notes'].dropna(),
    df['Base Notes'].dropna()
])

notes_list = all_notes.str.split(',').explode().str.strip().str.lower()

num_unique_notes = notes_list.nunique()
print(f"TOTAL NUMBER OF DIFFERENT NOTES : {num_unique_notes}\n")

most_common_note = notes_list.value_counts().idxmax()
most_common_count = notes_list.value_counts().max()
print(f"MOST FREQUENT NOTE : '{most_common_note}' ({most_common_count} occurrences)\n")

least_common_note = notes_list.value_counts().idxmin()
least_common_count = notes_list.value_counts().min()
print(f"LEAST FREQUENT NOTE : '{least_common_note}' ({least_common_count} occurrences)")

In [ ]:
top50 = df.sort_values('weighted_rating', ascending=False).head(50) # CORRECTION

for note_type in ['Top Notes', 'Middle Notes', 'Base Notes']:
    print(f"\nMOST COMMON {note_type} :")
    notes_series = top50[note_type].dropna().str.split(',').explode().str.strip()
    display(notes_series.value_counts().head(10))

In [ ]:
notes_list = (                                                  # Combine all note columns into a single list
    df[['Top Notes', 'Middle Notes', 'Base Notes']]
    .fillna('').agg(','.join, axis=1).str.split(',').explode().str.strip().replace('', np.nan).dropna()
)

notes_freq = notes_list.value_counts()                           # Then count most common notes
print("TOP 10 MOST COMMON NOTES :\n")
display(notes_freq.head(10))

In [ ]:
note_rank = notes_list.value_counts().rank(method='min', ascending=False).get('vanilla', None)
print(f"RANK OF 'vanilla' AMONG ALL NOTES : {int(note_rank) if note_rank else 'Not found'}\n")

In [ ]:
note_rank = notes_list.value_counts().rank(method='min', ascending=False).get('tuberose', None)
print(f"RANK OF 'tuberose' AMONG ALL NOTES : {int(note_rank) if note_rank else 'Not found'}\n")

In [ ]:
note_search = 'vanilla'

vanilla_perfumes = df[                                               # Filter all perfumes containing the note in any position (Top, Middle, Base)
    df[['Top Notes', 'Middle Notes', 'Base Notes']].apply(
        lambda x: x.str.lower().str.contains(note_search, na=False)
    ).any(axis=1)
]

print(f"TOTAL PERFUMES CONTAINING THE NOTE '{note_search}' : {len(vanilla_perfumes)}\n")

In [ ]:
note_search = 'tuberose'

tuberose_perfumes = df[                                              # Filter all perfumes containing the note in any position (Top, Middle, Base)
    df[['Top Notes', 'Middle Notes', 'Base Notes']].apply(
        lambda x: x.str.lower().str.contains(note_search, na=False)
    ).any(axis=1)
]

print(f"TOTAL PERFUMES CONTAINING THE NOTE '{note_search}' : {len(tuberose_perfumes)}\n")

In [ ]:
top_tuberose = tuberose_perfumes.sort_values(
    by='weighted_rating', ascending=False).head(5).round(3)

print("TOP 5 TUBEROSE PERFUMES (by WEIGHTED RATING) :\n")
display(top_tuberose[['Perfume', 'Brand', 'Rating Value', 'Rating Count', 'weighted_rating']])

In [ ]:
for col in ['Top Notes', 'Middle Notes', 'Base Notes']:
    subset = tuberose_perfumes[
        tuberose_perfumes[col].str.lower().str.contains(note_search, na=False)
    ].sort_values(by='weighted_rating', ascending=False).head(5).round(3)

    print(f"TOP 5 TUBEROSE PERFUMES BY {col} (by WEIGHTED RATING) :\n")
    display(subset[['Perfume', 'Brand', 'Rating Value', 'weighted_rating', col]])

In [ ]:
# Which main accords dominate the dataset?
accord_cols = ['mainaccord1','mainaccord2','mainaccord3','mainaccord4','mainaccord5']
all_accords = df[accord_cols].melt()['value'].str.lower().value_counts()
all_accords.head(10)

In [ ]:
df_notes = df[['Top Notes', 'Base Notes', 'weighted_rating']].dropna()

df_notes['NOTE COMBO'] = (
    df_notes['Top Notes'].str.split(',').str[0].str.strip() + " + " +
    df_notes['Base Notes'].str.split(',').str[0].str.strip()
)

combo_rating = (
    df_notes.groupby('NOTE COMBO')['weighted_rating']
    .mean().sort_values(ascending=False)
)

print("TOP 5 NOTE COMBOS (by WEIGHTED RATING) :\n")
display(combo_rating.head(5).round(3))      # best combos

In [ ]:
display(combo_rating.tail(5).round(3))       # worst combos

In [ ]:
mainaccord_cols = ['mainaccord1','mainaccord2','mainaccord3','mainaccord4','mainaccord5']
all_mainaccords = df[mainaccord_cols].apply(lambda x: x.str.lower().str.strip()).stack()
mainaccord_counts = all_mainaccords.value_counts()
print("TOP 20 MAIN ACCORDS :\n", mainaccord_counts.head(10))